# Pilot: recurring visual units in continuous ASL

Select a GPU runtime and choose **Runtime → Run all**. This notebook learns a 100-unit codebook from 5,000 unlabeled YouTube-ASL sentence clips in shard 1. It then tests (1) cluster reproducibility across disjoint source videos, (2) recurrence across clips/sources, and (3) strict leave-one-signer-out lexical-form consistency on 627 held-out ASL Citizen tokens. YouTube source IDs are not verified signer IDs; only ASL Citizen supports the cross-signer claim. The exploratory gate is a falsification tool, not a significance threshold or evidence of semantics.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU, then Run all again.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)


In [ ]:
import shutil, subprocess, sys
from pathlib import Path
if shutil.which('aria2c') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'aria2'], check=True)
REPO_URL = 'https://github.com/ss-sebastian/youtube-asl-skeleton-bert.git'
REPO_REF = 'agent/shape-aware-stgcn'
PROJECT = Path('/content/youtube-asl-skeleton-bert')
if PROJECT.exists(): shutil.rmtree(PROJECT)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(PROJECT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT), 'scikit-learn>=1.4'], check=True)
print('Project ready:', PROJECT)


## One required local upload
Upload `sign_unit_probe_inputs.zip`. It contains the already-trained vanilla contrastive checkpoint and the compact 30 MB ASL Citizen model-input evaluation set. It does **not** contain YouTube-ASL training data; shard 1 is downloaded temporarily from the official repository.


In [ ]:
from google.colab import files
target = Path('/content/sign_unit_probe_inputs.zip')
if not target.exists():
    uploaded = files.upload()
    assert uploaded, 'No bundle uploaded.'
    source = Path('/content') / next(iter(uploaded))
    if source != target: source.rename(target)
print('Input bundle:', target, f'{target.stat().st_size / 1024**2:.1f} MiB')


In [ ]:
import runpy
runpy.run_path(str(PROJECT / 'scripts/colab_unit_probe.py'), run_name='__main__')


In [ ]:
import json, pandas as pd
from IPython.display import display
result_root = Path('/content/sign_unit_probe/results')
continuous = json.loads((result_root / 'continuous_unit_metrics.json').read_text())
lexical = json.loads((result_root / 'lexical_unit_metrics.json').read_text())
gate = json.loads((result_root / 'pilot_gate.json').read_text())
display(pd.DataFrame([continuous]))
display(pd.DataFrame([lexical]))
display(pd.DataFrame([gate]))
recurrence = pd.read_csv(result_root / 'unit_recurrence.csv')
recurrence.plot.scatter(x='sources', y='clips', title='Unit recurrence across source videos', grid=True)
